#16 — Gold MERGE INTO: Late-Arriving Data Upserts

## Configuration

In [0]:
# Demo against fact_street_readings_liquid (16's output). Simulates:
#   - 500 "corrected" readings (existing rows with pollution recalculated)
#   - 10 "late-arriving" new readings (synthetic reading_id >= 900000 —
#     real reading_id values come from the source `id` column which cycles
#     0-998, so anything >= 900000 can never collide with real data and is
#     trivially identifiable/cleanable)
# Idempotent: previous demo rows are deleted by that same marker before
# each run, same as the reference project's cleanup step.

from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("gold_schema", "gold", "2. Gold Schema")
CATALOG = dbutils.widgets.get("catalog_name")
GOLD = dbutils.widgets.get("gold_schema")
TARGET = f"{CATALOG}.{GOLD}.fact_street_readings_liquid"

SYNTHETIC_DATE_FLOOR = "2025-01-01"   # marks late-arriving demo rows — real data ends 2024-03-11

## Cleanup previous demo rows

In [0]:
print(f"Cleaning up previous demo rows from {TARGET}...")
spark.sql(f"DELETE FROM {TARGET} WHERE reading_ts >= '{SYNTHETIC_DATE_FLOOR}'")
before_count = spark.table(TARGET).count()
print(f"Rows BEFORE merge: {before_count:,}")

## Build late-arriving batch: 500 updates + 10 new inserts

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW late_arriving_batch AS
SELECT * FROM (
    SELECT
        street_id, reading_date, reading_ts, reading_hour,
        noise,
        ROUND(pollution * 1.05, 6) AS pollution,   -- simulated correction: +5%
        light, raining, raining_clipped,
        bronze_load_dt, bronze_source, silver_load_dt, business_load_dt,
        current_timestamp() AS gold_load_dt
    FROM {TARGET}
    WHERE reading_ts < '{SYNTHETIC_DATE_FLOOR}'
    LIMIT 500
)
UNION ALL
SELECT * FROM (
    SELECT
        street_id,
        DATE('{SYNTHETIC_DATE_FLOOR}') + INTERVAL 1 DAY * rn AS reading_date,
        TIMESTAMP('{SYNTHETIC_DATE_FLOOR}') + INTERVAL 1 DAY * rn AS reading_ts,
        0 AS reading_hour,
        noise, pollution, light, raining, raining_clipped,
        bronze_load_dt, bronze_source, silver_load_dt, business_load_dt,
        current_timestamp() AS gold_load_dt
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY reading_ts) AS rn
        FROM {TARGET}
        WHERE reading_ts < '{SYNTHETIC_DATE_FLOOR}'
        LIMIT 10
    )
)
""")
print("Late-arriving batch view created (500 updates + 10 new inserts).")

## Execute MERGE

In [0]:
# ON street_id + reading_ts only — the confirmed 2-column grain for streets.
print(f"Executing MERGE INTO {TARGET}...")
spark.sql(f"""
MERGE INTO {TARGET} AS target
USING late_arriving_batch AS source
ON target.street_id = source.street_id
   AND target.reading_ts = source.reading_ts

WHEN MATCHED AND target.pollution != source.pollution THEN
  UPDATE SET
    target.pollution = source.pollution,
    target.gold_load_dt = source.gold_load_dt

WHEN NOT MATCHED THEN INSERT (
    street_id, reading_date, reading_ts, reading_hour,
    noise, pollution, light, raining, raining_clipped,
    bronze_load_dt, bronze_source, silver_load_dt, business_load_dt, gold_load_dt
) VALUES (
    source.street_id, source.reading_date, source.reading_ts, source.reading_hour,
    source.noise, source.pollution, source.light, source.raining, source.raining_clipped,
    source.bronze_load_dt, source.bronze_source, source.silver_load_dt, source.business_load_dt, source.gold_load_dt
)
""")

after_count = spark.table(TARGET).count()
print(f"\n{'='*60}")
print(f"  Rows BEFORE : {before_count:,}")
print(f"  Rows AFTER  : {after_count:,}")
print(f"  Net new     : {after_count - before_count:,}  (expected: 10)")
print(f"{'='*60}")

if after_count - before_count != 10:
    raise Exception(f"Expected exactly 10 net new rows, got {after_count - before_count}")
print("  MERGE verification PASSED")

display(spark.sql(f"SELECT street_id, reading_ts, pollution, gold_load_dt "
                   f"FROM {TARGET} WHERE reading_ts >= '{SYNTHETIC_DATE_FLOOR}' ORDER BY reading_ts"))

